# Extracting Data from an API into a DataFrame

**What is an API?**  
API = Application Programming Interface.  
It's a URL you hit → it sends back data (usually JSON) → you convert it into a DataFrame.

**Flow:**
```
Request URL  →  Get JSON Response  →  Convert to DataFrame
```

In [2]:
import requests
import pandas as pd
from io import StringIO

## 1. Simple GET Request

In [3]:
# free public api - no key needed
url = 'https://jsonplaceholder.typicode.com/users'

response = requests.get(url)
print('Status Code:', response.status_code)  # 200 means success

Status Code: 200


## 2. Understanding the Response

In [4]:
# raw text response
response.text[:300]

'[\n  {\n    "id": 1,\n    "name": "Leanne Graham",\n    "username": "Bret",\n    "email": "Sincere@april.biz",\n    "address": {\n      "street": "Kulas Light",\n      "suite": "Apt. 556",\n      "city": "Gwenborough",\n      "zipcode": "92998-3874",\n      "geo": {\n        "lat": "-37.3159",\n        "lng": "8'

In [5]:
# convert to python dict/list
data = response.json()
type(data)

list

In [6]:
# look at first record
data[0]

{'id': 1,
 'name': 'Leanne Graham',
 'username': 'Bret',
 'email': 'Sincere@april.biz',
 'address': {'street': 'Kulas Light',
  'suite': 'Apt. 556',
  'city': 'Gwenborough',
  'zipcode': '92998-3874',
  'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
 'phone': '1-770-736-8031 x56442',
 'website': 'hildegard.org',
 'company': {'name': 'Romaguera-Crona',
  'catchPhrase': 'Multi-layered client-server neural-net',
  'bs': 'harness real-time e-markets'}}

## 3. Convert JSON to DataFrame

In [7]:
df = pd.DataFrame(data)
df

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,"{'street': 'Norberto Crossing', 'suite': 'Apt....",1-477-935-8478 x6430,ola.org,"{'name': 'Considine-Lockman', 'catchPhrase': '..."
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,"{'street': 'Rex Trail', 'suite': 'Suite 280', ...",210.067.6132,elvis.io,"{'name': 'Johns Group', 'catchPhrase': 'Config..."
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,"{'street': 'Ellsworth Summit', 'suite': 'Suite...",586.493.6943 x140,jacynthe.com,"{'name': 'Abernathy Group', 'catchPhrase': 'Im..."
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,"{'street': 'Dayna Park', 'suite': 'Suite 449',...",(775)976-6794 x41206,conrad.com,"{'name': 'Yost and Sons', 'catchPhrase': 'Swit..."
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,"{'street': 'Kattie Turnpike', 'suite': 'Suite ...",024-648-3804,ambrose.net,"{'name': 'Hoeger LLC', 'catchPhrase': 'Central..."


In [8]:
df.shape

(10, 8)

In [9]:
df.columns

Index(['id', 'name', 'username', 'email', 'address', 'phone', 'website',
       'company'],
      dtype='str')

In [10]:
df.dtypes

id           int64
name           str
username       str
email          str
address     object
phone          str
website        str
company     object
dtype: object

## 4. Extracting Specific Columns

In [ ]:
df[['id', 'name', 'email', 'phone']]

## 5. Nested JSON → Flatten it

In [ ]:
# notice 'address' and 'company' are nested dicts inside each record
data[0]['address']

In [ ]:
# json_normalize flattens nested json into clean columns
from pandas import json_normalize

df_flat = json_normalize(data)
df_flat.head()

In [ ]:
df_flat.columns.tolist()

In [ ]:
# now address fields are their own columns
df_flat[['name', 'address.city', 'address.zipcode', 'company.name']]

## 6. Real World Example — Countries API

In [11]:
url = 'https://restcountries.com/v3.1/all'
response = requests.get(url)
print('Status:', response.status_code)
print('Total countries:', len(response.json()))

Status: 400
Total countries: 2


In [12]:

countries = response.json()

# look at one record
countries[0].keys()

KeyError: 0

In [13]:
# extract only what we need manually
records = []

for c in countries:
    records.append({
        'country'    : c.get('name', {}).get('common', 'N/A'),
        'capital'    : c.get('capital', ['N/A'])[0] if c.get('capital') else 'N/A',
        'region'     : c.get('region', 'N/A'),
        'population' : c.get('population', 0),
        'area_km2'   : c.get('area', 0),
        'currency'   : list(c.get('currencies', {}).keys())[0] if c.get('currencies') else 'N/A'
    })

df_countries = pd.DataFrame(records)
df_countries.head(10)

AttributeError: 'str' object has no attribute 'get'

In [ ]:
df_countries.shape

In [ ]:
# top 10 most populated countries
df_countries.sort_values('population', ascending=False).head(10)

In [ ]:
# filter only Asian countries
df_countries[df_countries['region'] == 'Asia']

## 7. API with Query Parameters

In [ ]:
# search only countries in Asia
url = 'https://restcountries.com/v3.1/region/asia'
response = requests.get(url)

asia = response.json()
print('Asian countries:', len(asia))

In [ ]:
# using params dict instead of building url manually
url = 'https://restcountries.com/v3.1/name/india'
response = requests.get(url)

india = response.json()
india[0]['name']

## 8. Handling API Errors

In [ ]:
url = 'https://restcountries.com/v3.1/name/xyzabc123'
response = requests.get(url)

if response.status_code == 200:
    df = pd.DataFrame(response.json())
    print(df)
else:
    print('Error:', response.status_code, response.text)

## 9. Save API Data to CSV

In [ ]:
df_countries.to_csv('countries.csv', index=False)
print('saved!')

In [ ]:
# read it back
pd.read_csv('countries.csv').head()